In [2]:
import argparse
import logging
from espnet2.bin.asr_inference import Speech2Text
import gc
import os
import csv
from speechbrain.inference.ASR import WhisperASR
from speechbrain.inference.ASR import EncoderASR
from utils.read_transcription import *
from utils.normalise_text import *
from pathlib import Path
from hyperpyyaml import load_hyperpyyaml
import librosa
import torch
from utils.meta import get_audio_info
from utils.apply_vad import *
from utils.list_files import list_files
from utils.VAD_chunk import *
from utils.wer_chunk import wer_chunk
from utils.logging_config import setup_logging
from utils.wer_segment import wer_segment
models = ["wav2vec","whisper-VAD-chunk","whisper-large","whisper-medium","whisper-large-VAD-chunk","wav2vec2-VAD-chunk"]
import gc
gc.collect()

ModuleNotFoundError: No module named 'espnet2'

In [2]:
wer_hparams = load_hyperpyyaml("""wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats""")


In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
w2v = EncoderASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-wav2vec2-commonvoice-fr", savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-wav2vec2-commonvoice-fr", run_opts={"device":"cuda:0"})
whisper_med = WhisperASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-medium-commonvoice-fr",savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-medium-commonvoice-fr", run_opts={"device":"cuda:0"})
whisper_large = WhisperASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-large-v2-commonvoice-fr",savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-large-v2-commonvoice-fr", run_opts={"device":"cuda:0"})

speech2text_ester = Speech2Text(
            "/home/rouas/experiments/SpeechRecognition/saved_models/espnet2-conformer-FR/asr_conformer_config.yaml",
            "/home/rouas/experiments/SpeechRecognition/saved_models/espnet2-conformer-FR/asr_conformer.pth",
            device="cuda:0"
        )
speech2text = Speech2Text(
            "/vol/experiments3/rouas/SpeechRecognition/saved_models/espnet2-commonvoice-conformer-FR/asr_commonvoice_conformer_FR_config.yaml",
            "/vol/experiments3/rouas/SpeechRecognition/saved_models/espnet2-commonvoice-conformer-FR/asr_commonvoice_conformer_FR.pth",
            device="cuda:0"
        )

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

speechbrain.integrations.huggingface.huggingface - Wav2Vec2Model is frozen.


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

speechbrain.integrations.huggingface.huggingface - WhisperModel is frozen.
speechbrain.integrations.huggingface.whisper - whisper encoder-decoder is frozen.


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

speechbrain.integrations.huggingface.huggingface - WhisperModel is frozen.
speechbrain.integrations.huggingface.whisper - whisper encoder-decoder is frozen.


In [51]:
wav_data="/vol/corpora/Daoudi/Data/Reading/PD" 
ref_trans= "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/Transcript_Monologue_final/PD"
txt = "/vol/corpora/Daoudi/Data/text_chevre.txt"
csv_path = "khalid/khalid_PD_read.csv"
pred_folder = "khalid"

In [13]:
def list_files(trans_dir):
    tg_to_wav = {}
    for f in sorted(os.listdir(trans_dir)):
        for w in os.listdir(wav_data):
            if f.endswith(".txt") and not w.endswith("old"):
                if f.split("-")[1].split(".")[0] in w:
                    tg_to_wav[f] = w
           

    return tg_to_wav
tg_to_wav = list_files(ref_trans)
tg_to_wav

{'1PD-AGJI.txt': '1PD-AGJI-chevre.wav',
 '1PD-ASTW.txt': '1PD-ASTW-chevre.wav',
 '1PD-BGRG.txt': '1PD-BGRG-chevre.wav',
 '1PD-BHEW.txt': '1PD-BHEW-chevre.wav',
 '1PD-BSQI.txt': '1PD-BSQI-chevre.wav',
 '1PD-DAAY.txt': '1PD-DAAY-chevre.wav',
 '1PD-DKCM.txt': '1PD-DKCM-chevre.wav',
 '1PD-FDOD.txt': '1PD-FDOD-chevre.wav',
 '1PD-FSKS.txt': '1PD-FSKS-chevre.wav',
 '1PD-HVQV.txt': '1PD-HVQV-chevre.wav',
 '1PD-IXVU.txt': '1PD-IXVU-chevre_bis.wav',
 '1PD-KTTU.txt': '1PD-KTTU-chevre.wav',
 '1PD-LDBC.txt': '1PD-LDBC-chevre.wav',
 '1PD-LEZQ.txt': '1PD-LEZQ-chevre_bis.wav',
 '1PD-NHVW.txt': '1PD-NHVW-chevre.wav',
 '1PD-OFAS.txt': '1PD-OFAS-chevre.wav',
 '1PD-OZNC.txt': '1PD-OZNC-chevre.wav',
 '1PD-OZSZ.txt': '1PD-OZSZ-chevre.wav',
 '1PD-PXMX.txt': '1PD-PXMX-chevre.wav',
 '1PD-UVLI.txt': '1PD2-UVLI-chevre.wav',
 '1PD-WVGL.txt': '1PD-WVGL-chevre.wav',
 '1PD-XFZE.txt': '1PD2-XFZE-chevre.wav',
 '1PD-YIWI.txt': '1PD-YIWI-chevre.wav',
 '1PD-ZNPV.txt': '1PD-ZNPV-chevre.wav',
 '1PD-ZPHY.txt': '1PD-ZPHY-che

In [14]:
len(tg_to_wav)

32

In [8]:
import re

def clean_transcription(text):
    # 1. Garder seulement la partie après ***
    if "***" in text:
        text = text.split("***", 1)[1]

    # 2. Supprimer contenu entre [] et ()
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\(.*?\)", "", text)

    # 3. Supprimer # mais garder contenu
    text = text.replace("#", "")

    # 4. Nettoyage des espaces multiples
    text = re.sub(r"\s+", " ", text)

    # 5. Nettoyage des espaces en début/fin de lignes
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    return (" ".join(lines)).split(" ")




In [9]:
with open(txt,"r") as f:
    t=f.read()


In [18]:
import os
import csv
import torch
from hyperpyyaml import load_hyperpyyaml
from speechbrain.utils.metric_stats import ErrorRateStats
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# ===================== WER CONFIG =====================
wer_hparams = load_hyperpyyaml("""
wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats
""")

# ===================== WER FUNCTION =====================
def wer_chunk(results, words):
    hyp = ""
    for r in results:
        hyp += r["text"] + " "
    ref = " ".join(words)

    hyp_norm = normalization(hyp)
    ref_norm = normalization(ref)

    wer_hparams["wer_stats"].clear()
    wer_hparams["wer_stats"].append(
        ids=[0],
        predict=[hyp_norm],
        target=[ref_norm]
    )

    stats = wer_hparams["wer_stats"].summarize()

    S = stats["substitutions"]
    D = stats["deletions"]
    I = stats["insertions"]
    WER = stats["WER"]

    print(f'WER={WER:.4f}, S={S}, D={D}, I={I}')

    return ref_norm, hyp_norm, WER, S, D, I


# ===================== CORPUS STATS =====================
corpus_stats = {
    "w2vec": {"S": 0, "D": 0, "I": 0, "N": 0},
    "whisper": {"S": 0, "D": 0, "I": 0, "N": 0},
    "whisper_large": {"S": 0, "D": 0, "I": 0, "N": 0},
    "conf_cv": {"S": 0, "D": 0, "I": 0, "N": 0},
    "conf_ester": {"S": 0, "D": 0, "I": 0, "N": 0},
    "hmm": {"S": 0, "D": 0, "I": 0, "N": 0},
}

# ===================== CSV =====================
with open(csv_path, "w", newline="", encoding="utf-8") as f:

    fieldnames = [
        "filename", "duration_sec", "samplerate", "channels",

        "trans_w2vec_vad_chunk", "WER_w2vec_vad_chunk", "S_w2vec", "D_w2vec", "I_w2vec",
        "trans_whisper_vad_chunk", "WER_whisper_vad_chunk", "S_whisper", "D_whisper", "I_whisper",
        "trans_whisper_large_vad_chunk", "WER_whisper_large_vad_chunk", "S_whisper_large", "D_whisper_large", "I_whisper_large",
        "trans_conf_cv_vad_chunk", "WER_conf_cv_vad_chunk", "S_conf_cv", "D_conf_cv", "I_conf_cv",
        "trans_conf_ester_vad_chunk", "WER_conf_ester_vad_chunk", "S_conf_ester", "D_conf_ester", "I_conf_ester",
        "trans_hmm_tdnn_vad_chunk", "WER_hmm_tdnn_vad_chunk", "S_hmm", "D_hmm", "I_hmm"
    ]

    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    # ===================== LOOP =====================
    #for tg, wav in tg_to_wav.items():
    for wav in sorted(os.listdir(wav_data)):
        wav_file = os.path.join(wav_data, wav)
        #trans_file = os.path.join(ref_trans, tg)
        if not wav.endswith(".old"):
            #if not (os.path.exists(wav_file) and os.path.exists(trans_file)):
                #continue

            
            info = get_audio_info(wav_file)
            row = {"filename": wav, **info}
    
            # Load audio
            audio_np, sr = read_audio_16k(wav_file)
            wav_tensor = torch.from_numpy(audio_np)
    
            # VAD
            chunks = vad_chunk_with_timestamps(wav_tensor)
    
            # Reference
            
            words = clean_transcription(t)
    
            ref_len = len(normalization(" ".join(words)))
    
            # ======== W2VEC ========
            results = whisper_transcribe_chunks(w2v, "wav2vec2-VAD-chunk", wav_tensor, chunks)
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_w2vec_vad_chunk"] = pred
            row["WER_w2vec_vad_chunk"] = wer
            row["S_w2vec"], row["D_w2vec"], row["I_w2vec"] = S, D, I
    
            corpus_stats["w2vec"]["S"] += S
            corpus_stats["w2vec"]["D"] += D
            corpus_stats["w2vec"]["I"] += I
            corpus_stats["w2vec"]["N"] += ref_len
    
            # ======== WHISPER MED ========
            results = whisper_transcribe_chunks(whisper_med, "whisper-VAD-chunk", wav_tensor, chunks)
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_whisper_vad_chunk"] = pred
            row["WER_whisper_vad_chunk"] = wer
            row["S_whisper"], row["D_whisper"], row["I_whisper"] = S, D, I
    
            corpus_stats["whisper"]["S"] += S
            corpus_stats["whisper"]["D"] += D
            corpus_stats["whisper"]["I"] += I
            corpus_stats["whisper"]["N"] += ref_len
    
            # ======== WHISPER LARGE ========
            results = whisper_transcribe_chunks(whisper_large, "whisper-large-VAD-chunk", wav_tensor, chunks)
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_whisper_large_vad_chunk"] = pred
            row["WER_whisper_large_vad_chunk"] = wer
            row["S_whisper_large"], row["D_whisper_large"], row["I_whisper_large"] = S, D, I
    
            corpus_stats["whisper_large"]["S"] += S
            corpus_stats["whisper_large"]["D"] += D
            corpus_stats["whisper_large"]["I"] += I
            corpus_stats["whisper_large"]["N"] += ref_len
    
            # ======== CONFORMER CV ========
            
            results = espnet_transcribe_chunks(speech2text, wav_tensor, chunks, sr=16000)
    
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_conf_cv_vad_chunk"] = pred
            row["WER_conf_cv_vad_chunk"] = wer
            row["S_conf_cv"], row["D_conf_cv"], row["I_conf_cv"] = S, D, I
    
            corpus_stats["conf_cv"]["S"] += S
            corpus_stats["conf_cv"]["D"] += D
            corpus_stats["conf_cv"]["I"] += I
            corpus_stats["conf_cv"]["N"] += ref_len
    
            # ======== CONFORMER ESTER ========
            
            results = espnet_transcribe_chunks(speech2text_ester, wav_tensor, chunks, sr=16000)
    
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_conf_ester_vad_chunk"] = pred
            row["WER_conf_ester_vad_chunk"] = wer
            row["S_conf_ester"], row["D_conf_ester"], row["I_conf_ester"] = S, D, I
    
            corpus_stats["conf_ester"]["S"] += S
            corpus_stats["conf_ester"]["D"] += D
            corpus_stats["conf_ester"]["I"] += I
            corpus_stats["conf_ester"]["N"] += ref_len
    
            # ======== HMM-TDNN ========
            results = hmmtdnn_transcribe_chunks(
                "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/asr_FR_kaldi_hmm_tdnn.sh",
                wav_tensor, chunks,
                "/vol/experiments3/imbenamor/TAPAS-FRAIS/logs/transcription/rhap",
                sr=16000
            )
    
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_hmm_tdnn_vad_chunk"] = pred
            row["WER_hmm_tdnn_vad_chunk"] = wer
            row["S_hmm"], row["D_hmm"], row["I_hmm"] = S, D, I
    
            corpus_stats["hmm"]["S"] += S
            corpus_stats["hmm"]["D"] += D
            corpus_stats["hmm"]["I"] += I
            corpus_stats["hmm"]["N"] += ref_len
    
            writer.writerow(row)


# ===================== FINAL CORPUS RESULTS =====================
print("\n===== FINAL CORPUS WER =====")

for model, stats in corpus_stats.items():
    S, D, I, N = stats["S"], stats["D"], stats["I"], stats["N"]

    wer = (S + D + I) / N if N > 0 else 0

    print(f"{model}: WER={wer:.4f} | S={S}, D={D}, I={I}, N={N}")

WER=8.4507, S=6, D=0, I=0


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


WER=21.1268, S=13, D=1, I=1
WER=12.6761, S=9, D=0, I=0
WER=15.4930, S=11, D=0, I=0
WER=16.9014, S=11, D=1, I=0
WER=16.9014, S=12, D=0, I=0
WER=14.0845, S=7, D=0, I=3
WER=21.1268, S=13, D=1, I=1
WER=19.7183, S=11, D=2, I=1
WER=25.3521, S=14, D=1, I=3
WER=19.7183, S=11, D=0, I=3
WER=26.7606, S=13, D=1, I=5
WER=9.8592, S=6, D=1, I=0
WER=29.5775, S=13, D=7, I=1
WER=35.2113, S=8, D=17, I=0
WER=14.0845, S=10, D=0, I=0
WER=8.4507, S=6, D=0, I=0
WER=21.1268, S=13, D=0, I=2
WER=18.3099, S=10, D=3, I=0
WER=26.7606, S=16, D=2, I=1
WER=28.1690, S=16, D=1, I=3
WER=21.1268, S=15, D=0, I=0
WER=26.7606, S=17, D=0, I=2
WER=22.5352, S=14, D=2, I=0
WER=9.8592, S=6, D=1, I=0
WER=49.2958, S=11, D=23, I=1
WER=39.4366, S=8, D=17, I=3
WER=19.7183, S=14, D=0, I=0
WER=15.4930, S=11, D=0, I=0
WER=23.9437, S=13, D=1, I=3
WER=14.0845, S=9, D=1, I=0
WER=19.7183, S=13, D=0, I=1
WER=12.6761, S=8, D=0, I=1
WER=9.8592, S=7, D=0, I=0
WER=14.0845, S=9, D=0, I=1
WER=21.1268, S=13, D=0, I=2
WER=19.7183, S=11, D=2, I=1
WER=

# Phoneme eval

In [1]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
from transformers import AutoModelForCTC, Wav2Vec2Processor
import json
from tqdm import tqdm
import os
from textgrid import TextGrid
import editdistance
from utils.VAD_chunk import *
from utils.metrics import *

/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

MODEL_ID = "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wav2vec2-french-phonemizer"

model = AutoModelForCTC.from_pretrained(MODEL_ID)
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)

device = "cpu"
model = model.to(device)
model.eval()

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [3]:
import torch, librosa
from transformers import WavLMForCTC, Wav2Vec2FeatureExtractor, Wav2Vec2PhonemeCTCTokenizer

CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme/checkpoint-268600"
ROOT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme"          # where vocab.json / tokenizer were saved

device = "cuda"
model = WavLMForCTC.from_pretrained(CKPT).to(device).eval()
feat  = Wav2Vec2FeatureExtractor.from_pretrained(ROOT)   # also present in CKPT
tok   = Wav2Vec2PhonemeCTCTokenizer.from_pretrained(ROOT)

model = model.to(device)
model.eval()

WavLMForCTC(
  (wavlm): WavLMModel(
    (feature_extractor): WavLMFeatureEncoder(
      (conv_layers): ModuleList(
        (0): WavLMGroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x WavLMNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x WavLMNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): WavLMFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): WavLMEncoder(
      (p

In [47]:
#txt = "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/Transcription_Reading4Labri/Transcription_chevre_Margot.txt"
txt = "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/Transcript_Monologue_final/MSA/"
#wav_data="/vol/corpora/Daoudi/Data/Reading/HC/"
wav_data = "/vol/corpora/Daoudi/Data/Monologue/MSA"
ref_files = os.listdir(txt)
os.listdir(wav_data),ref_files

(['2MSA-TDHV-image.wav',
  '1MSA-HGJK-image.wav',
  '1MSA-ESWQ-image.wav',
  '2MSA2-IZHO-image.wav',
  '2MSA2-WCGQ-image.wav',
  '1MSA-YWWF-image.wav',
  '2MSA-LGIS_image.wav',
  '2MSA-ZQMQ-image.wav',
  '1MSA2-MIHM-image.wav',
  '2MSA-BDID-image.wav',
  '1MSA2-HQZQ-image.wav',
  '1MSA-FXAY-image.wav',
  '1MSA-EDKI-image.wav',
  '2MSA2-VUWZ-image.wav',
  '2MSA2-GCKC-image.wav',
  '1MSA-TJHD-image.wav',
  '1MSA-JAJC-image.wav',
  '2MSA-PBLQ-image.wav',
  '1MSA-PLZK-image.wav',
  '1MSA2-KHDK-image.wav',
  '1MSA2-PDTD-image.wav',
  '1MSA2-ADOD-image.wav',
  '2MSA2-BQME-image.wav',
  '1MSA-MCKC-image.wav',
  '2MSA2-VSCS-image.wav',
  '2MSA-FTCM-image.wav',
  '2MSA-DWSN-image.wav'],
 ['1MSA-ADOD.txt',
  '1MSA-EDKI.txt',
  '1MSA-PLZK.txt',
  '1MSA-PDTD.txt',
  '2MSA-BDID.txt',
  '2MSA-VUWZ.txt',
  '2MSA-WCGQ.txt',
  '2MSA-LGIS.txt',
  '2MSA-TDHV.txt',
  '1MSA-HGJK.txt',
  '1MSA-MIHM.txt',
  '1MSA-JAJC.txt',
  '1MSA-HQZQ.txt',
  '2MSA-VSCS.txt',
  '1MSA-KHDK.txt',
  '2MSA-IZHO.txt',
  '2MSA-B

In [15]:
"""
WhisperX-style VAD chunking for the Whisper-encoder + CTC phoneme model.

Built on your approach: uses whisperx.vads.pyannote.load_vad_model, reads the
RAW per-frame VAD scores, and binarizes them with WhisperX's onset/offset
thresholds. The goal is unchanged from the original vad_chunk_with_timestamps:
return a list of {"start", "end"} (seconds, original time) chunks, each <= 30 s,
ready for the inference loop.

Two stages:
  1. VAD + binarize  -> WhisperX raw scores -> speech segments (hysteresis: go
                        active above `onset`, inactive below `offset`).
  2. cut & merge     -> pack segments into <= chunk_size windows, KEEPING internal
                        pauses inside a window. A single segment longer than
                        chunk_size is split at its QUIETEST frame (real WhisperX
                        behaviour, possible here because we kept the raw scores),
                        so nothing ever hits Whisper's 30 s truncation.

Only load_whisperx_vad() touches whisperx, so the rest is importable/testable
offline without it.
"""

import numpy as np
import torch
from whisperx.vads.pyannote import load_vad_model
# WhisperX defaults
ONSET = 0.5
OFFSET = 0.363


def load_whisperx_vad(wav):
    vad_pipeline = load_vad_model(
    device="cuda",
    token=os.environ["HF_TOKEN"])
    vad_scores = vad_pipeline(wav)
    scores = vad_scores.data[:, 0]
    frames = vad_scores.sliding_window
    times = [frames[i].middle for i in range(len(scores))]
    return scores, times

def _binarize(scores, times, onset=ONSET, offset=OFFSET):
    """Hysteresis binarization -> list of (start_s, end_s) speech segments."""
    segments = []
    is_active = scores[0] > onset
    start = times[0] if is_active else None
    for t, sc in zip(times[1:], scores[1:]):
        if is_active:
            if sc < offset:
                segments.append((start, t))
                is_active = False
        else:
            if sc > onset:
                start = t
                is_active = True
    if is_active:
        segments.append((start, times[-1]))
    return segments


def _split_long_by_score(seg_start, seg_end, times, scores, chunk_size):
    """Split a > chunk_size segment at the quietest frame in each window.

    Mirrors WhisperX: when a segment exceeds max_duration, cut at the lowest
    detection score in the second half rather than at a hard time boundary, so
    the cut lands on minimally-active speech.
    """
    pieces, cur = [], seg_start
    while seg_end - cur > chunk_size:
        lo, hi = cur + chunk_size * 0.5, cur + chunk_size
        idx = np.where((times >= lo) & (times <= hi))[0]
        cut = times[idx[np.argmin(scores[idx])]] if len(idx) else cur + chunk_size
        pieces.append((cur, cut))
        cur = cut
    pieces.append((cur, seg_end))
    return pieces


def merge_chunks(segments, times, scores, chunk_size=20.0):
    """WhisperX cut & merge -> list of {"start","end"} chunks, each <= chunk_size."""
    times = np.asarray(times, dtype=float)
    scores = np.asarray(scores, dtype=float)
    split = []
    for s, e in segments:
        if e - s > chunk_size:
            split.extend(_split_long_by_score(s, e, times, scores, chunk_size))
        else:
            split.append((s, e))

    if not split:
        return []

    merged = []
    curr_start, curr_end = split[0]
    for s, e in split[1:]:
        if e - curr_start > chunk_size and curr_end - curr_start > 0:
            merged.append({"start": curr_start, "end": curr_end})
            curr_start = s
        curr_end = e
    merged.append({"start": curr_start, "end": curr_end})
    return merged


def vad_chunk_with_timestamps(
    wav,
    sampling_rate=16000,
    max_chunk_duration=8.0,
    onset=ONSET,
    offset=OFFSET,
):
    """Drop-in replacement. Same return type as the old rVAD version.

    wav               : torch.Tensor (1D, 16 kHz)  -- or a file path
    vad_model         : object from load_whisperx_vad() (load it once, reuse it)
    max_chunk_duration: keep <= 30.0 for Whisper's hard cap
    """
    scores, times = load_whisperx_vad(wav)
    segments = _binarize(scores, times, onset, offset)
    return merge_chunks(segments, times, scores, chunk_size=max_chunk_duration)
def get_phonemes(model, processor, audio_path):
    audio, sr = sf.read(audio_path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    wav = torch.from_numpy(audio)
    print(len(wav)/16000)
    chunks = vad_chunk_with_timestamps(audio_path)
    device = next(model.parameters()).device
    blank_id = model.config.pad_token_id

    full_phoneme_parts = []

    for chunk in chunks:
        start_sample = int(chunk["start"] * 16000)
        end_sample   = int(chunk["end"]   * 16000)
        chunk_tensor = wav[start_sample:end_sample]
        
        inputs = processor(
            chunk_tensor.numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits

        predicted_ids = torch.argmax(logits, dim=-1)[0]
        
        decoded = processor.batch_decode(predicted_ids.unsqueeze(0))[0]
        full_phoneme_parts.append(decoded.strip())

        num_frames     = logits.shape[1]
        chunk_duration = (end_sample - start_sample) / 16000
        print(chunk_duration)
        frame_duration = chunk_duration / num_frames

        # ✅ Reset per chunk
        prev_id = None
        current_alignment = None
        for frame_idx, token_id in enumerate(predicted_ids.tolist()):
    
            if token_id == blank_id:
                prev_id = token_id
                continue
            
            phoneme = processor.decode([token_id])
            
            # Skip empty / space / word separator
            if phoneme in ["", " ", "|"]:
                prev_id = token_id
                continue
            
            # If this is a combining mark → attach to previous phoneme
            if unicodedata.combining(phoneme):
                if current_alignment is not None:
                    current_alignment["phoneme"] += phoneme
                prev_id = token_id
                continue
            
            prev_id = token_id

    full_phoneme_string = " ".join(full_phoneme_parts)
    return full_phoneme_string.strip()
    
        

In [6]:
def get_phonemes_wavlm(model, feat,tok, audio_path):
    audio, sr = sf.read(audio_path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    wav = torch.from_numpy(audio)
    print(len(wav)/16000)
    chunks = vad_chunk_with_timestamps(audio_path,max_chunk_duration=30)
    device = next(model.parameters()).device
    blank_id = model.config.pad_token_id

    full_phoneme_parts = []

    for chunk in chunks:
        start_sample = int(chunk["start"] * 16000)
        end_sample   = int(chunk["end"]   * 16000)
        chunk_tensor = wav[start_sample:end_sample]
        
        inputs = feat(            # was: processor(
        chunk_tensor.numpy(),
        sampling_rate=16000,
            return_tensors="pt",
            )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits

        predicted_ids = torch.argmax(logits, dim=-1)[0]
        
        decoded = tok.batch_decode(predicted_ids.unsqueeze(0))[0]
        full_phoneme_parts.append(decoded.strip())

        num_frames     = logits.shape[1]
        chunk_duration = (end_sample - start_sample) / 16000
        print(chunk_duration)
        frame_duration = chunk_duration / num_frames

        # ✅ Reset per chunk
        prev_id = None
        current_alignment = None
        for frame_idx, token_id in enumerate(predicted_ids.tolist()):
    
            if token_id == blank_id:
                prev_id = token_id
                continue
            
            phoneme = tok.decode([token_id])
            
            # Skip empty / space / word separator
            if phoneme in ["", " ", "|"]:
                prev_id = token_id
                continue
            
            # If this is a combining mark → attach to previous phoneme
            if len(phoneme) == 1 and unicodedata.combining(phoneme):
                if current_alignment is not None:
                    current_alignment["phoneme"] += phoneme
                prev_id = token_id
                continue
            
            prev_id = token_id

    full_phoneme_string = " ".join(full_phoneme_parts)
    return full_phoneme_string.strip()
    
        

In [7]:
import re
import unicodedata

# Put here the phonemes that are allowed to be nasalized in your inventory
ALLOWED_NASAL_BASES = {"a", "ɑ", "ɛ", "ɔ"}

COMBINING_TILDE = "\u0303"


def ipa_tokenize(text, allowed_nasal_bases=ALLOWED_NASAL_BASES):
    # Normalize so combining marks are represented consistently
    text = unicodedata.normalize("NFD", text)

    tokens = []
    i = 0

    while i < len(text):
        char = text[i]

        if char.isspace():
            i += 1
            continue

        # If the character is followed by combining marks, collect them
        if i + 1 < len(text) and unicodedata.combining(text[i + 1]) != 0:
            j = i + 1
            marks = []
            while j < len(text) and unicodedata.combining(text[j]) != 0:
                marks.append(text[j])
                j += 1

            marks_str = "".join(marks)

            # Keep nasalization only for allowed bases
            if COMBINING_TILDE in marks_str:
                if char in allowed_nasal_bases:
                    tokens.append(char + COMBINING_TILDE)
                else:
                    tokens.append(char)   # drop the tilde
            else:
                tokens.append(char + marks_str)

            i = j
        else:
            tokens.append(char)
            i += 1

    return tokens


def clean_and_tokenize(text, allowed_nasal_bases=ALLOWED_NASAL_BASES):
    # 1. Ignore everything before ***
    if "***" in text:
        text = text.split("***", 1)[1]

    # 2. Remove content inside [] and ()
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\(.*?\)", "", text)

    # 3. Remove # but keep content
    text = text.replace("#", "")

    # 4. Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()
    text=re.sub(r':', r'', text)
    text=re.sub(r'[\r\n]+', '', text)
    text=re.sub(r'/', '', text)
    # 5. Tokenize
    phonemes = ipa_tokenize(text, allowed_nasal_bases=allowed_nasal_bases)
    phonemes = [i.replace("ɑ","a") for i in phonemes]
    phonemes = [i.replace("õ","ɔ̃") for i in phonemes]
        
    # 6. Join with single space
    normalized = " ".join(phonemes)

    # 7. Unique set
    phoneme_set = sorted(set(phonemes))

    return normalized, phoneme_set

In [8]:
def write_trn_from_corpus(ref_dict, hyp_dict,
                         ref_path="ref.trn",
                         hyp_path="hyp.trn"):

    common_utts = sorted(set(ref_dict.keys()) & set(hyp_dict.keys()))

    with open(ref_path, "w", encoding="utf-8") as f_ref, \
         open(hyp_path, "w", encoding="utf-8") as f_hyp:

        for utt_id in common_utts:
            ref_seq = ref_dict[utt_id]
            hyp_seq = hyp_dict[utt_id]

            # convert list → string
            if isinstance(ref_seq, list):
                ref_seq = " ".join(ref_seq)
            if isinstance(hyp_seq, list):
                hyp_seq = " ".join(hyp_seq)

            # normalize spaces
            ref_seq = " ".join(ref_seq.strip().split())
            hyp_seq = " ".join(hyp_seq.strip().split())

            f_ref.write(f"{ref_seq} ({utt_id})\n")
            f_hyp.write(f"{hyp_seq} ({utt_id})\n")

In [48]:
mapp={}
for j in sorted(ref_files):
    if not j.startswith(".ip"):
        for i in os.listdir(wav_data):
            if j.split(".")[0].split("-")[1] in i and not i.startswith(".") and not i.endswith("old"):
                #if j=="1PD2-OAAY.txt":
                    #mapp["1PD2-OAAY.txt"]="1PD2-OAYY-chevre.wav"
                #if j.split("-")[1].split(".")[0] in i and not i.endswith("old"):
    
                mapp[j]=i
            if j =="1MSA-MIHM.txt":
                #mapp["1MSA-MIHM.txt"] = '1MSA2-MIMH-chevre.wav'
                mapp["1MSA-MIHM.txt"] = '1MSA2-MIHM-image.wav'
            if j=='2MSA-VSCS.txt':
                #mapp['2MSA-VSCS.txt'] = '2MSA2-VCSC-chevre_bis.wav'
                mapp['2MSA-VSCS.txt'] = '2MSA2-VSCS-image.wav'

In [13]:
with open(txt,"r") as f:
    ref_phonemes=f.read()
ref_phonemes

IsADirectoryError: [Errno 21] Is a directory: '/vol/experiments3/imbenamor/TAPAS-FRAIS/data/Transcript_Monologue_final/MSA/'

In [49]:
import unicodedata as ud

import re
import unicodedata as ud


def clean_spont(file):
    with open(file, encoding="utf-8") as f:
        text = f.read()

    # Garder uniquement avant ***
    text = text.split("***")[0]

    # Nettoyage
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\(.*?\)", "", text)
    text = text.replace("#", "")
    text = text.replace(":", "")
    text = text.replace("/", "")

    # Normalisation Unicode
    text = ud.normalize("NFD", text)

    # =========================================================
    # CORRECTION DES NASALES
    # =========================================================

    # ɶ̃ -> ɛ̃
    text = re.sub(r"ɶ̃", "ɛ̃", text)

    # ̃ɶ -> ɛ̃
    text = re.sub(r"̃ɶ", "ɛ̃", text)

    # ̃ ɶ -> ɛ̃
    text = re.sub(r"̃\s+ɶ", "ɛ̃", text)

    # Normaliser les espaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenisation par mot
    words = text.split()

    PHON_MAP = {
        "õ": "ɔ̃",
        "ɑ": "a",
        "ɑ̃": "ã",
        "ε": "ɛ",
        "ε̃": "ɛ̃",
        "ɶ": "œ",
        "œ̃": "ɛ̃",

        "ʀ": "ʁ",
        "r": "ʁ",
        "x": "ʁ",

        "ɪ": "i",
        "ʊ": "u",
        "g": "ɡ",
    }

    # Normalisation Unicode du dictionnaire
    PHON_MAP = {
        ud.normalize("NFD", k): ud.normalize("NFD", v)
        for k, v in PHON_MAP.items()
    }

    def norm(p):
        p = ud.normalize("NFD", p)
    
        p = (
            p.replace("ː", "")
             .replace(":", "")
             .strip(".,;!?")
        )
    
        # Le tilde seul n'est pas un phonème
        if p == "̃":
            return None
    
        return PHON_MAP.get(p, p) if p else None
    normalized_words = []

    for word in words:

        # =====================================================
        # CAS PARTICULIER : ɛ̃
        # =====================================================
        # On ne passe PAS ɛ̃ dans ipa_tokenize(),
        # car ipa_tokenize() semble supprimer le tilde.
        if word == "ɛ̃":
            phonemes = ["ɛ̃"]

        else:
            phonemes = ipa_tokenize(word)

        # Normalisation
        phonemes = [norm(p) for p in phonemes]
        phonemes = [p for p in phonemes if p]

        normalized_words.append(" ".join(phonemes))

    normalized = " ".join(normalized_words)

    return normalized


In [45]:
import librosa
import unicodedata
import re
from collections import defaultdict
ref_dict={}
hyp_dict={}
for wav in os.listdir(wav_data):
    if not wav.endswith("old") and not wav.startswith("."):
        audio_path = os.path.join(wav_data, wav)
        print(audio_path)
        pred_phonemes,pred_s = clean_and_tokenize(get_phonemes_wavlm(model, feat,tok, audio_path))
        #key = next((k for k, v in mapp.items() if v == wav), None)
        #ref_phonemes = clean_spont(txt+key)
        #print(ref_phonemes)
        ref_phonemes,_ = clean_and_tokenize(ref_phonemes)
        
        ref_dict[wav.split(".")[0].split("-")[1]]=ref_phonemes
        hyp_dict[wav.split(".")[0].split("-")[1]]=pred_phonemes
        

Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


/vol/corpora/Daoudi/Data/Monologue/HC/1HC-IQKL-image.wav
92.125375
19.524375
19.895625
19.2375
26.949375
6.51375
/vol/corpora/Daoudi/Data/Monologue/HC/1HC2-TATW-image.wav


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


103.5546875
25.92
26.139375
29.93625
17.28
/vol/corpora/Daoudi/Data/Monologue/HC/1HC-ZANM-image.wav
107.196


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


...

27.995625
/vol/corpora/Daoudi/Data/Monologue/HC/1HC-PLIQ-image.wav
97.165375
18.984375
20.89125
15.103125
25.498125
16.47
/vol/corpora/Daoudi/Data/Monologue/HC/1HC2-YSZL-image.wav


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


56.701375


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


28.704375
27.23625
/vol/corpora/Daoudi/Data/Monologue/HC/2HC-IWHW-image.wav
45.7539375
21.33
24.418125


In [50]:

#spon,
import librosa
import unicodedata
import re
from collections import defaultdict
ref_dict={}
hyp_dict={}
for wav in os.listdir(wav_data):
    if not wav.endswith("old") and not wav.startswith("."):
        audio_path = os.path.join(wav_data, wav)
        print(audio_path)
        #pred_phonemes,pred_s = clean_and_tokenize(get_phonemes_wavlm(model, feat,tok, audio_path))
        pred_phonemes,pred_s = clean_and_tokenize(get_phonemes(model, processor, audio_path))

        key = next((k for k, v in mapp.items() if v == wav), None)
        ref_phonemes = clean_spont(txt+key)
        print(ref_phonemes)
        #ref_phonemes,_ = clean_and_tokenize(ref_phonemes)
        
        ref_dict[wav.split(".")[0].split("-")[1]]=ref_phonemes
        hyp_dict[wav.split(".")[0].split("-")[1]]=pred_phonemes
        

Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


/vol/corpora/Daoudi/Data/Monologue/MSA/2MSA-TDHV-image.wav
50.926875
6.98625
6.1425
6.6825
7.543125
4.876875
6.21


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


1.805625
s e t i m a ʒ d e k ʁ i y n y n i l v i z i b l m ã œ p ɔ ʁ v ʁ e z ã b l ə m ã ɔ̃ i v w a o p ʁ ə m j e p l ã œ b a t o k i ʁ ə s ã b l ə a y n ɡ ʁ u i n ɛ t t ʁ w a m a a v ɛ k œ s ɛ ʁ t ɛ̃ n ɔ̃ b ʁ d ə p a t i l e b a t o t u t o t u ʁ ã d œ z j ɛ m p l ã œ k ɛ ã t ʁ w a z j ɛ m p l ã y n v i l a v ɛ k d e z i m ø b l ə v i l m ɔ d ɛ ʁ n i l j a k ã m ɛ m y n s ɔ ʁ t d ə d ɔ̃ ʒ ɔ̃ s y ʁ l a s y ʁ l a ɡ o ʃ k i l ɛ s ã t ʁ ə v w a ʁ y n v i l œ p ø p l y œ p ø p l y p a s e
/vol/corpora/Daoudi/Data/Monologue/MSA/1MSA-HGJK-image.wav
51.372
6.969375
5.63625
7.644375
0.624375
5.97375
7.4925


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


...

6.5475
4.1175
4.2525
7.02
3.661875
3.1725
5.4
7.12125
7.05375
6.37875
5.923125
4.01625
5.4675
4.01625
7.99875
4.3875
6.51375
5.67
7.3575


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


5.011875
0.57375
n u z a l ɔ̃ ʁ e a l i z e o ʒ u ʁ d ɥ i l a p a t a ʃ u a l ɔ ʁ p u ʁ l a p a t a ʃ u i l f o d ə l a f a ʁ i n d ə l o u d y l ɛ u d y l ɛ e l o d y b œ ʁ d y s y k ʁ ə d y s ɛ l d ɔ̃ k l e k ã t i t e s e d ø s ã s ɛ̃ k ã t ɡ ʁ a m d ə l i k i d s ã k a ʁ ã t ɡ ʁ a m d ə f a ʁ i n s ã d i ɡ ʁ a m d ə b œ ʁ t ʁ w a t ʁ w a ɡ ʁ a m d ə s ɛ l e k ɛ̃ z ɡ ʁ a m d ə s y k ʁ ə ɔ̃ m ɛ d ã y n k a s ʁ o l l ə l e u l ə l i k i d a v ɛ k l ə b œ ʁ l ə s ɛ l u l ə s y k ʁ ə a l ɔ ʁ a ʁ i v e a e b y l i s j ɔ̃ a ʁ i v e a e b y l i s j ɔ̃ ɔ̃ ʁ ə t i ʁ l a k a s ʁ o l d y f ø e ɔ̃ ʒ ɛ t l a f a ʁ i n l e s ã k a ʁ ã t ɡ ʁ a m d œ s œ l k u ɔ̃ p ʁ ã y n s p a t y l e ɔ̃ m e l ã ʒ l ə l ə l ã s ã b l ə l ə l i k i d e e l a f a ʁ i n e k ã s a d ə v j ɛ̃ o m ɔ ʒ ɛ n ɔ̃ ʁ ə m e s y ʁ l ə f ø e ɔ̃ t u ʁ n a v ɛ k l a s p a t y l p u ʁ d e s e ʃ e l a p a t y n f w a k ə l a p a t e d e s e ʃ e ɔ̃ l ə v w a o f ɔ̃ d ə l a k a s ʁ o l i l j a œ ʁ e z i d y ɔ̃ m e s ɛ t 

Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


5.315625
m ɛ ʒ u ʁ n e e m ɛ n ɥ i s ɔ̃ k ɔ̃ f ɔ̃ d y ʒ ə m ə l ɛ v t ʁ ɛ t o ʒ ə t ʁ ɛ n d ã l a m ɛ z ɔ̃ ã b a ʒ ə v a k ʒ ə v a ʒ ə v j ɛ̃ ʒ e œ i d e ã t ɛ t ʒ e p a s j e b ɛ t m ã d ə l a t e l e v i z j ɔ̃ ʒ a v a l t u t l a ʒ u ʁ n e ʒ ə n e ɔ k ɛ̃ o k y n p a s j ɔ̃ ʒ ə s y b i p a ʁ l a t e l e k ɔ m ã d m ə f a t i ɡ t ɛ t i s t a i z
/vol/corpora/Daoudi/Data/Monologue/MSA/2MSA-DWSN-image.wav
93.208
5.011875
4.86
7.99875
5.956875
6.766875
6.31125
6.935625
6.91875
5.518125
3.358125
3.74625
3.40875
5.16375
3.5775
o ʒ u ʁ d ɥ i ʃ ɛ ʁ z a m i l a ʁ ə s ɛ t d ə l œ f o p l a ʃ ɛ ʁ ʃ e y n p u l ʃ ɛ ʁ ʃ e œ p l a k u ʁ e a p ʁ e l a p u l ʃ ɛ ʁ ʃ e l œ f t ʁ u v e l œ f œ œ f b j ɛ̃ f ʁ e œ œ f d ə ʃ e n u p ʁ e n e l œ f k a s e l œ f e a t ã s j ɔ̃ d ã l ə p l a f ɛ t k u l e l b l ã f ɛ t k u l e l ʒ o n m e l ã ʒ e p a l ʒ o n e l b l ã s a l e p w a v ʁ e f ɛ t k ɥ i ʁ d u s ə m ã ʒ ã t i m ã p a ʁ l e l ɥ i ã m ɛ m t ã p ɥ i s ɛ ʁ v e ʃ o œ b ɔ̃ œ f d ə ʃ e n

In [51]:
s1 = set()
for i in ref_dict.values():
    s1.update(i.split())

In [52]:

_RAW = {
    "õ": "ɔ̃", "ɑ̃": "ã","ε": "ɛ", "ε̃": "ɛ̃",
    "ɑ": "a", "ʀ": "ʁ", "r": "ʁ", "x": "ʁ",
    "ɪ": "i", "ʊ": "u", "g": "ɡ",
}
PHON_MAP = {ud.normalize("NFD", k): ud.normalize("NFD", v) for k, v in _RAW.items()}

def norm(p):
    p = ud.normalize("NFD", p).replace("ː", "").replace(":", "").strip(".,;!?")
    return PHON_MAP.get(p, p) if p else None

def norm_seq(seq):
    """Pour une chaîne déjà tokenisée (pred décodé, ref existante)."""
    out = [norm(p) for p in seq.split()]
    return " ".join(p for p in out if p)
hyp_dict = {k: norm_seq(v) for k, v in hyp_dict.items()}
s = set()
for i in hyp_dict.values():
    s.update(i.split())

In [53]:
s1-s

{'ɥ'}

In [54]:
s-s1

{'ɨ', 'ɾ', '̃', 'β'}

In [55]:
write_trn_from_corpus(ref_dict, hyp_dict,
                         ref_path="khalid/ref_MSA_wav2vec_newvad.trn",
                         hyp_path="khalid/hyp_MSA_wav2vec_newvad.trn")

In [56]:
!/home/rouas/Sources/git/kaldi/tools/sctk/bin/sclite \
  -r khalid/ref_MSA_wav2vec_newvad.trn trn \
  -h khalid/hyp_MSA_wav2vec_newvad.trn trn \
   -i wsj \
  -o all 


sclite: 2.10 TK Version 1.3
Begin alignment of Ref File: 'khalid/ref_MSA_wav2vec_newvad.trn' and Hyp File: 'khalid/hyp_MSA_wav2vec_newvad.trn'
    Alignment# 1 for speaker ado          
    Alignment# 1 for speaker bdi          
    Alignment# 1 for speaker bqm          
    Alignment# 1 for speaker dws          
    Alignment# 1 for speaker edk          
    Alignment# 1 for speaker esw          
    Alignment# 1 for speaker ftc          
    Alignment# 1 for speaker fxa          
    Alignment# 1 for speaker gck          
    Alignment# 1 for speaker hgj          
    Alignment# 1 for speaker hqz          
    Alignment# 1 for speaker izh          
    Alignment# 1 for speaker jaj          
    Alignment# 1 for speaker khd          
    Alignment# 1 for speaker lgi          
    Alignment# 1 for speaker mck          
    Alignment# 1 for speaker mih          
    Alignment# 1 for speaker pbl          
    Alignment# 1 for speaker pdt          
    Alignment# 1 for speaker plz        

In [57]:
def align(ref, hyp):
    n, m = len(ref), len(hyp)
    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # deletion
                    dp[i][j-1],    # insertion
                    dp[i-1][j-1]   # substitution
                )

    # backtrace
    i, j = n, m
    aligned_ref, aligned_hyp = [], []

    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
            aligned_ref.append(ref[i-1])
            aligned_hyp.append(hyp[j-1])
            i -= 1; j -= 1

        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            aligned_ref.append(ref[i-1])
            aligned_hyp.append("<del>")
            i -= 1

        elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
            aligned_ref.append("<ins>")
            aligned_hyp.append(hyp[j-1])
            j -= 1

        else:
            aligned_ref.append(ref[i-1])
            aligned_hyp.append(hyp[j-1])
            i -= 1; j -= 1

    return aligned_ref[::-1], aligned_hyp[::-1]
def load_trn(path):
    sequences = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            text = line[:line.rfind("(")].strip()
            tokens = text.split()
            sequences.append(tokens)
    return sequences

In [58]:
from collections import defaultdict, Counter
import pandas as pd

def evaluate(ref_file, hyp_file):
    ref_seqs = load_trn(ref_file)
    hyp_seqs = load_trn(hyp_file)

    confusion = defaultdict(lambda: defaultdict(int))
    ref_counts = Counter()
    hyp_counts = Counter()

    S = D = I = N = 0

    for ref, hyp in zip(ref_seqs, hyp_seqs):
        aligned_ref, aligned_hyp = align(ref, hyp)

        for r, h in zip(aligned_ref, aligned_hyp):
            
            if r != "<ins>":
                ref_counts[r] += 1
                N += 1
            if h != "<del>":
                hyp_counts[h] += 1

            if r == h:
                confusion[r][r] += 1  
            else:
                if r == "<ins>":
                    I += 1
                elif h == "<del>":
                    D += 1
                else:
                    S += 1
                    confusion[r][h] += 1

    PER = (S + D + I) / N

    return {
        "PER": PER,
        "S": S,
        "D": D,
        "I": I,
        "confusion": confusion,
        "ref_counts": ref_counts,
        "hyp_counts": hyp_counts
    }

In [59]:
def build_confusion_df(confusion, ref_counts, hyp_counts):
    import pandas as pd

    # 🔥 ADD IT HERE
    phonemes = sorted(set(ref_counts.keys()) | set(hyp_counts.keys()))

    # create square matrix
    df = pd.DataFrame(0, index=phonemes, columns=phonemes)

    # fill with counts
    for r in confusion:
        for h in confusion[r]:
            df.loc[r, h] += confusion[r][h]

    return df
def phoneme_metrics(conf_df):
    metrics = []

    for p in conf_df.index:
        TP = conf_df.loc[p, p]
        FN = conf_df.loc[p].sum() - TP
        FP = conf_df[p].sum() - TP

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        metrics.append({
            "phoneme": p,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

    return pd.DataFrame(metrics)

In [60]:
import pandas as pd
from collections import defaultdict

def phoneme_stats(ref_seqs, hyp_seqs):
    stats = defaultdict(lambda: {
        "Cor": 0,
        "Sub": 0,
        "Del": 0,
        "Ins": 0,
        "Nombre": 0
    })

    for ref, hyp in zip(ref_seqs, hyp_seqs):
        aligned_ref, aligned_hyp = align(ref, hyp)

        for r, h in zip(aligned_ref, aligned_hyp):

            if r != "<ins>":
                stats[r]["Nombre"] += 1

            if r == h:
                if r != "<ins>":
                    stats[r]["Cor"] += 1

            elif r == "<ins>":
                stats[h]["Ins"] += 1

            elif h == "<del>":
                stats[r]["Del"] += 1

            else:
                stats[r]["Sub"] += 1

    # convert to dataframe
    rows = []
    for p, s in stats.items():
        N = s["Nombre"] if s["Nombre"] > 0 else 1

        rows.append({
            "phoneme": p,
            "Cor": s["Cor"],
            "Sub": s["Sub"],
            "Del": s["Del"],
            "Ins": s["Ins"],
            "Nombre": s["Nombre"],
            "%Cor": s["Cor"] / N,
            "%Sub": s["Sub"] / N,
            "%Del": s["Del"] / N,
            "%Ins": s["Ins"] / N
        })

    df = pd.DataFrame(rows).sort_values("Nombre", ascending=False)
    return df

In [61]:
res = evaluate("khalid/ref_MSA_wav2vec_newvad.trn", "khalid/hyp_MSA_wav2vec_newvad.trn")
ref_seqs = load_trn("khalid/ref_MSA_wav2vec_newvad.trn")
hyp_seqs = load_trn("khalid/hyp_MSA_wav2vec_newvad.trn")

df = phoneme_stats(ref_seqs, hyp_seqs)

df.to_csv("khalid/phoneme_stats_MSA_wav2vec_newvad.csv", index=False)
print(df.head())
conf_df = build_confusion_df(
    res["confusion"],
    res["ref_counts"],
    res["hyp_counts"]
)
conf_df.to_csv("khalid/confusion_MSA_wav2vec_newvad.csv")

metrics_df = phoneme_metrics(
conf_df
)
metrics_df.to_csv("khalid/phoneme_metrics_MSA_wav2vec_newvad.csv")

print("PER:", res["PER"])

   phoneme  Cor  Sub  Del  Ins  Nombre      %Cor      %Sub      %Del      %Ins
10       a  883   45   38   41     966  0.914079  0.046584  0.039337  0.042443
1        e  522  260   68   37     850  0.614118  0.305882  0.080000  0.043529
4        ʁ  700   51   49   41     800  0.875000  0.063750  0.061250  0.051250
18       l  554   58   46   50     658  0.841945  0.088146  0.069909  0.075988
14       t  538   20   19   44     577  0.932409  0.034662  0.032929  0.076256
PER: 0.23344536288491236


In [62]:
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion_heatmap(conf_df, normalize=True, save_path="khalid/confusion_heatmap_MSA_wav2vec_newvad.png"):
    df = conf_df.copy()

    if normalize:
        df = df.div(df.sum(axis=1), axis=0).fillna(0)

    plt.figure(figsize=(12, 10))
    plt.imshow(df.values, interpolation='nearest')
    plt.colorbar()

    plt.xticks(range(len(df.columns)), df.columns, rotation=90)
    plt.yticks(range(len(df.index)), df.index)

    plt.xlabel("Predicted phoneme")
    plt.ylabel("Reference phoneme")
    plt.title("Phoneme Confusion Matrix")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

In [63]:
# Heatmap
plot_confusion_heatmap(conf_df)




In [74]:
wav_data="/vol/corpora/Daoudi/Data/Reading/HC/" 

spk=[]
for i in os.listdir(wav_data):
    spk.append(i.split("-")[1])

In [75]:
maps={}
for s in spk:
    maps[s[:3]]=s
maps

{'FCS': 'FCSY',
 'IAJ': 'IAJC',
 'GJT': 'GJTD',
 'AEO': 'AEOW',
 'OZT': 'OZTK',
 'ODZ': 'ODZI',
 'KWZ': 'KWZA',
 'YSZ': 'YSZL',
 'MZQ': 'MZQO',
 'TAT': 'TATW',
 'ZAN': 'ZANM',
 'AQA': 'AQAC',
 'OZU': 'OZUD',
 'MLS': 'MLSV',
 'ECK': 'ECKC',
 'DWW': 'DWWQ',
 'LKZ': 'LKZT',
 'ACB': 'ACBC',
 'PLI': 'PLIQ',
 'KGA': 'KGAP',
 'IQK': 'IQKL',
 'PVS': 'PVSZ',
 'FXR': 'FXRA',
 'AVG': 'AVGE',
 'GDZ': 'GDZI',
 'CEZ': 'CEZE',
 'KXR': 'KXRA',
 'MLL': 'MLLK',
 'RKH': 'RKHI',
 'IWV': 'IWVU',
 'IWH': 'IWHW'}

In [77]:
import os
from collections import defaultdict

import os
from collections import defaultdict

def split_pra_by_speaker(pra_file, output_dir="khalid/speakers_pra_HC"):
    os.makedirs(output_dir, exist_ok=True)

    speakers = defaultdict(list)
    current_block = []
    current_spk = None

    with open(pra_file, "r", encoding="utf-8") as f:
        for line in f:

            # détecter début bloc
            if line.startswith("Speaker sentences"):
                # sauvegarder bloc précédent
                if current_block and current_spk:
                    speakers[current_spk].append("".join(current_block))
                    current_block = []

                # extraire speaker
                # exemple ligne:
                # Speaker sentences   0:  agj   #utts: 1
                parts = line.split()
                current_spk = parts[3]  # <-- agj

            # accumuler lignes
            if current_spk:
                current_block.append(line)

        # dernier bloc
        if current_block and current_spk:
            speakers[current_spk].append("".join(current_block))

    # écrire fichiers
    for spk, blocks in speakers.items():
        with open(os.path.join(output_dir, f"{maps[spk.upper()]}.pra"), "w", encoding="utf-8") as f:
            for b in blocks:
                f.write(b)

    print(f"{len(speakers)} speakers extracted")
# usage
split_pra_by_speaker("khalid/hyp_HC_wavlm.trn.pra")

31 speakers extracted
